In [ ]:
# 0) Installs
!pip install -q transformers peft trl datasets accelerate bitsandbytes seaborn scikit-learn pandas


In [ ]:
# ============================================================
# PropInsight — QLoRA Finetune FIRST (no baseline)
# Splits must already exist in /content/drive/MyDrive/PropInsight/propinsight_datasets
# Saves adapters to models/finetuned and predictions to results/finetuned_predictions.json
# ============================================================


# 1) Imports & Drive
import os, json, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, torch
from pathlib import Path
from tqdm.auto import tqdm
from datasets import Dataset
from sklearn.metrics import confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments, Trainer, EarlyStoppingCallback
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from google.colab import drive

drive.mount('/content/drive')

# 2) Paths (Path-only)
BASE = Path("/content/drive/MyDrive/PropInsight")

DATASETS_DIR = BASE / "propinsight_datasets"
TRAIN_JSON   = DATASETS_DIR / "train.json"
VAL_JSON     = DATASETS_DIR / "validation.json"
TEST_JSON    = DATASETS_DIR / "test.json"

MODELS_DIR         = BASE / "models"
FINETUNED_DIR      = MODELS_DIR / "finetuned"
RESULTS_DIR        = BASE / "results"
VISUALIZATIONS_DIR = BASE / "visualizations"
TENSORBOARD_LOGS   = RESULTS_DIR / "tensorboard"

FINETUNED_PREDICTIONS = RESULTS_DIR / "finetuned_predictions.json"
TRAINING_CONFIG       = RESULTS_DIR / "training_config.json"
EVAL_RESULTS          = RESULTS_DIR / "evaluation_results.json"  # finetuned-only summary

for d in [MODELS_DIR, FINETUNED_DIR, RESULTS_DIR, VISUALIZATIONS_DIR, TENSORBOARD_LOGS]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Dirs ready")
print(f"Splits present? train={TRAIN_JSON.exists()} val={VAL_JSON.exists()} test={TEST_JSON.exists()}")

# 3) Load existing splits
if not (TRAIN_JSON.exists() and VAL_JSON.exists() and TEST_JSON.exists()):
    raise FileNotFoundError("Missing split(s). Ensure train/validation/test.json exist under propinsight_datasets.")

train_data = json.load(open(TRAIN_JSON, "r", encoding="utf-8"))
val_data   = json.load(open(VAL_JSON,   "r", encoding="utf-8"))
test_data  = json.load(open(TEST_JSON,  "r", encoding="utf-8"))
print(f"✓ Loaded: train={len(train_data)} | val={len(val_data)} | test={len(test_data)}")

# 4) QLoRA setup (AISG base IT model)
MODEL_ID = "aisingapore/Qwen-SEA-LION-v4-32B-IT"   # base model for QLoRA

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
)
model.config.use_cache = False  # req. for gradient checkpointing
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# 5) Prompt building (teacher forcing via chat template)
SYSTEM_MSG = (
    "You are PropInsight, a Singapore real-estate expert. "
    "Return ONLY the Output block with fields: Overall Sentiment, Price Sentiment, "
    "Policy Sentiment, Affordability Sentiment, Location, Aspect, Entity, Policy Mentioned, "
    "Singlish Detected, Cultural Context, Emotion, Datetime, Source, Reasoning."
)

def sft_text(sample: dict) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",
         "content": f"{sample.get('instruction','Analyze the sentiment of this Singapore property comment:')}\n\n"
                    f"Input: {sample.get('input','')}\n\nReturn only the Output block."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt + "\nOutput: " + sample.get("output","")

train_ds = Dataset.from_list([{"text": sft_text(x)} for x in train_data])
val_ds   = Dataset.from_list([{"text": sft_text(x)} for x in val_data])

def tok_fn(batch):
    t = tokenizer(batch["text"], max_length=2048, truncation=True, padding="max_length")
    t["labels"] = t["input_ids"].copy()
    return t

train_ds = train_ds.map(tok_fn, batched=True, remove_columns=["text"])
val_ds   = val_ds.map(tok_fn,   batched=True, remove_columns=["text"])

# 6) TrainingArguments — early stopping + checkpoints + TB logs
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir=str(FINETUNED_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,

    eval_strategy="steps",       # ← correct key
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,

    logging_steps=5,
    logging_dir=str(TENSORBOARD_LOGS),
    report_to=["tensorboard"],

    bf16=bf16_ok,
    fp16=not bf16_ok,

    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    remove_unused_columns=False,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

early_stop = EarlyStoppingCallback(early_stopping_patience=3)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    callbacks=[early_stop],
)

# Save training config snapshot
json.dump({
    "model_id": MODEL_ID,
    "bnb_4bit": {"quant_type":"nf4","compute_dtype":"bfloat16","double_quant":True},
    "lora": {"r":16,"alpha":32,"dropout":0.05,"targets":["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]},
    "epochs": args.num_train_epochs,
    "batch_size": args.per_device_train_batch_size,
    "grad_accum": args.gradient_accumulation_steps,
    "lr": args.learning_rate
}, open(TRAINING_CONFIG, "w"), indent=2)

# 7) Train
trainer.train()
trainer.save_model(str(FINETUNED_DIR))
tokenizer.save_pretrained(str(FINETUNED_DIR))
print(f"✓ Saved LoRA adapters → {FINETUNED_DIR}")

# 8) Inference with finetuned adapters (on TEST_JSON)
def build_prompt_infer(text: str) -> str:
    messages = [
        {"role":"system","content":SYSTEM_MSG},
        {"role":"user","content":f"Analyze the sentiment of this Singapore property comment:\n\n{text}\n\nReturn only the Output block."}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def gen_output(mdl, tok, text: str) -> str:
    p = build_prompt_infer(text)
    inp = tok(p, return_tensors="pt", truncation=True, max_length=2048).to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inp,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
        )
    return tok.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True)

finetuned_preds = []
for item in tqdm(test_data, total=len(test_data), desc="Finetuned inference"):
    txt = item.get("input","")
    pred = gen_output(model, tokenizer, txt) if isinstance(txt, str) and txt.strip() else ""
    finetuned_preds.append({
        "input": txt, "ground_truth": item.get("output",""),
        "prediction": pred, "metadata": item.get("metadata", {})
    })

json.dump(finetuned_preds, open(FINETUNED_PREDICTIONS, "w"), indent=2, ensure_ascii=False)
print(f"✓ Saved predictions → {FINETUNED_PREDICTIONS}")

# 9) (Optional) quick finetuned-only overview metric
def extract_field(block: str, key: str, default: str = "neutral") -> str:
    m = re.search(rf"{re.escape(key)}\s*:\s*(.*)", block or "", re.IGNORECASE)
    return (m.group(1).strip().lower() if m else default)

y_true = [extract_field(p["ground_truth"], "Overall Sentiment") for p in finetuned_preds]
y_pred = [extract_field(p["prediction"],   "Overall Sentiment") for p in finetuned_preds]
labels = sorted(set(y_true) | set(y_pred))
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
metrics = {
    "samples": len(finetuned_preds),
    "overall_sentiment_accuracy": float(accuracy_score(y_true, y_pred)) if len(y_true) else 0.0,
    "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
    "macro_precision": float(precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
    "macro_recall": float(recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
    "labels": labels,
}
json.dump(metrics, open(EVAL_RESULTS, "w"), indent=2)
print("✓ Finetuned-only metrics saved:", EVAL_RESULTS)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dirs ready
Splits present? train=True val=True test=True
✓ Loaded: train=2825 | val=314 | test=554


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

trainable params: 134,217,728 || all params: 32,896,340,992 || trainable%: 0.4080


Map:   0%|          | 0/2825 [00:00<?, ? examples/s]

Map:   0%|          | 0/314 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
50,0.321000,0.342473
100,0.276000,0.327733
150,0.275400,0.325328


Step,Training Loss,Validation Loss
50,0.321000,0.342473
100,0.276000,0.327733
150,0.275400,0.325328


✓ Saved LoRA adapters → /content/drive/MyDrive/PropInsight/models/finetuned


Finetuned inference:   0%|          | 0/554 [00:00<?, ?it/s]